# Proyecto final integrador · Fase 4

## Factores sociodemográficos y su asociación con factores de riesgo cardiovascular, utilizando la Encuesta Nacional de Salud (ENS) 2016-2017 de Chile.

**MCDI500 · Programación para la Ciencia de Datos · Grupo 4**

Integrantes: Felipe Díaz Toro, Ninoska Yévenes Hernández, Guillermo Subiabre Chacón, Pablo Ríos Passteni
Docente: Omar Salinas

---
## Parte 0 · Entorno y reproducibilidad

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")

print("REGISTRO DEL ENTORNO")
print("=" * 58)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  NumPy       : {np.__version__}")
print(f"  pandas      : {pd.__version__}")
print(f"  Matplotlib  : {plt.matplotlib.__version__}")
print(f"  seaborn     : {sns.__version__}")
print(f"  Entorno virtual activo: {sys.prefix != sys.base_prefix}")

REGISTRO DEL ENTORNO
  Python      : 3.13.9
  NumPy       : 2.5.3
  pandas      : 3.0.5
  Matplotlib  : 3.11.1
  seaborn     : 0.13.2
  Entorno virtual activo: True


---
# I. Fase 1 · El problema y los objetivos

## 1.1 Contexto y problema

Las enfermedades cardiovasculares son la principal causa de muerte en Chile y en el mundo.
Según la Encuesta Nacional de Salud (ENS) 2016-2017 del Ministerio de Salud, la prevalencia
nacional de hipertensión arterial alcanza 27,6% y la de sospecha de diabetes mellitus 12,3%
en personas de 15 años y más (MINSAL, 2017). Estas cifras agregadas no informan cómo se
distribuye el riesgo al interior de la población.

**Problema.** Se dispone de registros individuales de la ENS 2016-2017, pero no de una
caracterización de cómo el perfil sociodemográfico de las personas encuestadas se asocia
con los principales indicadores de riesgo cardiovascular disponibles en la encuesta.

## 1.2 Pregunta de investigación

> ¿Cómo se relacionan las condiciones sociodemográficas (edad, sexo, nivel educacional,
> ingreso del hogar y zona de residencia) con la presencia de hipertensión arterial,
> diabetes, colesterol alto, índice de masa corporal y nivel de actividad física en los
> participantes de la ENS 2016-2017?

## 1.3 Objetivo general

> Caracterizar la relación entre las condiciones sociodemográficas de los participantes
> (edad, sexo, nivel educacional, ingreso del hogar y zona de residencia) y la presencia de
> hipertensión arterial, diabetes, colesterol alto, un índice de masa corporal elevado y un
> nivel insuficiente de actividad física en la ENS 2016-2017

## 1.4 Objetivos específicos

1. Comparar la prevalencia de hipertensión arterial entre los distintos grupos etarios.
2. Describir cómo varía la prevalencia de diabetes según el grupo etario y la zona de residencia.
3. Analizar la relación entre el nivel educacional y la presencia de colesterol alto.
4. Examinar la asociación entre el índice de masa corporal y el nivel de actividad física.
5. Evaluar si la relación entre edad e índice de masa corporal, en presencia de hipertensión, se mantiene al estratificar por sexo.

---
# II. Fase 2 · El conjunto preparado

La Fase 2 no se repite acá: se carga su resultado y se documentan las decisiones que lo
produjeron.

## 2.1 Decisiones de preparación y su justificación

| Decisión | Alternativa descartada | Razón |
|---|---|---|
| Conservar los registros sin ingreso (`as27`/`as28`), marcados con bandera | Eliminar las filas | Los faltantes no son aleatorios |
| Imputar `as27` por mediana dentro de su propio tramo (`as28`) | Imputar con la mediana global | Preserva la estructura por tramo de ingreso |
| Codificar Sexo, Zona, diabetes y colesterol con one-hot | Codificación ordinal | Son categorías nominales, sin orden |
| Escalar Edad, IMC e ingreso con media y desviación | Dejar las unidades originales | Requerido por Fase 3 |
| Reclasificar GPAQ de nominal a ordinal | Tratarla como nominal | Bajo < Moderado < Alto tiene orden |

**Consecuencia para la Fase 4.** Las dos últimas decisiones dejan el conjunto ilegible para
graficar (edad e IMC centrados en cero, sexo convertido en columnas de ceros y unos). La
Parte IV resuelve esto.

## 2.2 Carga del conjunto de la Fase 2

In [2]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent.parent  # ajusta si tu notebook no está en F4/notebooks/
sys.path.insert(0, str(RAIZ))

from src.carga import leer_archivo

datos = leer_archivo(str(RAIZ / "data" / "processed" / "ens_variables_f1f2.xlsx"))
print(f"Conjunto de origen (F1-F2): {datos.shape[0]} filas x {datos.shape[1]} columnas")
datos.head(3)

Conjunto de origen (F1-F2): 5520 filas x 16 columnas


,IdEncuesta,FechaInicioF1,Edad,Sexo,Zona,HTA,di3,dis2,IMC,anos_estudio_MINSAL_1,GPAQ,as27,as28,Fexp_F1F2p_Corr,Conglomerado,Estrato
0,20008,2016-11-27 15:10:40,25,2,1,0.0,2,3,23.198208,15.0,3.0,200000.0,3,52.307503,13101101,131
1,20011,2016-11-26 15:52:49,20,2,1,0.0,2,3,26.203983,14.0,3.0,NaN,-9999,250.753357,13101102,131
2,20012,2016-11-20 13:16:39,85,2,1,1.0,2,2,23.732346,15.0,1.0,1300000.0,10,133.128342,13101102,131


---
# III. Fase 3 · El núcleo algorítmico

Se integran las clases de `src/` que ya construyeron y se verifica que el pipeline
reproduce exactamente el resultado oficial de la Fase 2.

## 3.1 El pipeline completo

In [3]:
from src.imputadores import MarcadorNoRespuesta, EliminadorFilasNulas, ImputadorFlexible
from src.imputadores import PorMediana, PorMedianaDeTramo, PorModa
from src.transformadores import CodificadorOneHot, EscaladorEstandar, EliminadorColumna, ConvertidorEntero
from src.pipeline import Pipeline

variables_categoricas = ["Sexo", "Zona", "di3", "dis2"]
variables_numericas = ["Edad", "IMC", "as27"]

pasos_limpieza = [
    MarcadorNoRespuesta("as28"),
    EliminadorFilasNulas("HTA"),
    ImputadorFlexible("IMC", PorMediana()),
    ImputadorFlexible("anos_estudio_MINSAL_1", PorMediana()),
    ImputadorFlexible("as27", PorMedianaDeTramo("as28")),
    ImputadorFlexible("GPAQ", PorModa()),
]

pasos_integracion = (
    [EliminadorColumna(col) for col in ["IdEncuesta", "FechaInicioF1"]]
    + [ConvertidorEntero(col) for col in ["HTA", "GPAQ"]]
    + [CodificadorOneHot(col) for col in variables_categoricas]
    + [EscaladorEstandar(col) for col in variables_numericas]
)

pipeline_completo = Pipeline(pasos_limpieza + pasos_integracion)
df_transformado = pipeline_completo.ajustar_transformar(datos)

print(f"Dimensiones iniciales : {datos.shape}")
print(f"Dimensiones finales   : {df_transformado.shape}")

Dimensiones iniciales : (5520, 16)
Dimensiones finales   : (5511, 23)


### 3.2 Eficiencia computacional

Además de validar el pipeline, se comprueba brevemente el costo computacional de los transformadores propios frente a una implementación de referencia de scikit-learn/pandas. El análisis completo (variación de tamaño de muestra, medición de memoria con `tracemalloc`, y evaluación de complejidad) se realizó en Fase 3 y está documentado en `F3/notebooks/S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb`. Aquí se replica la comparación de tiempos sobre el conjunto real de este proyecto (5.511 filas), como evidencia directa dentro del notebook consolidado.

In [5]:
from src.medicion import medir_tiempo_timeit
from sklearn.preprocessing import StandardScaler

def escalar_propio(df, columna):
    return EscaladorEstandar(columna).ajustar_transformar(df[[columna]].copy())

def escalar_sklearn(df, columna):
    resultado = df[[columna]].copy()
    resultado[columna] = StandardScaler().fit_transform(resultado[[columna]])
    return resultado

tiempo_propio = medir_tiempo_timeit(escalar_propio, df_visual, "Edad")
tiempo_sklearn = medir_tiempo_timeit(escalar_sklearn, df_visual, "Edad")

pd.DataFrame([
    {"implementacion": "EscaladorEstandar (propio)", "tiempo_s": tiempo_propio},
    {"implementacion": "StandardScaler (sklearn)", "tiempo_s": tiempo_sklearn},
])

NameError: name 'df_visual' is not defined

## 3.2 Verificación contra el resultado oficial de la Fase 2

In [ ]:
from src.validador import verificar_contra_fase2

RUTA_PROCESADO = RAIZ / "data" / "processed" / "ens_procesado.csv"

pipeline_verificacion = Pipeline(pasos_limpieza + pasos_integracion)
coincide = verificar_contra_fase2(
    pipeline_verificacion,
    datos,
    RUTA_PROCESADO,
    verificar_tipos=True,
)
assert coincide, "El pipeline no reproduce el resultado oficial de la Fase 2"
print("Verificado: el pipeline reproduce exactamente ens_procesado.csv")

[OK] El resultado coincide con la referencia: 5511 filas x 23 columnas comparadas
Verificado: el pipeline reproduce exactamente ens_procesado.csv


## 3.3 Validación técnica (59 pruebas)

Casos normal, límite y excepción, definidos en `tests/casos_por_clase.py`, ejecutados
directamente acá.

In [ ]:
from tests.casos_por_clase import ejecutar_pruebas

reporte = ejecutar_pruebas()
print(reporte.to_string(index=False))

59 pruebas: 59 OK, 0 con problemas
               clase escenario                                                                              prueba estado                                                                                                             detalle
       Transformador    normal        Ajustar devuelve el mismo paso, guarda lo aprendido y transformar lo aplica.     OK                                                                                                                    
       Transformador    limite              parametros entrega una copia: modificarla no altera el estado interno.     OK                                                                                                                    
       Transformador    limite         transformar trabaja sobre una copia y deja intacto el DataFrame de entrada.     OK                                                                                                                    
       Transf

---
# IV. Fase 4 · Conjunto interpretable para visualizar

`df_transformado` tiene la edad e IMC centrados en cero y las categóricas en one-hot —
correcto para el análisis, ilegible para graficar. La solución: capturar el dato en el
punto del pipeline donde ya está limpio pero todavía no se codifica ni escala, justo antes
de `pasos_integracion`. Se reutilizan las mismas clases, sin duplicar lógica de limpieza.

## 4.1 df_visual

In [ ]:
pipeline_limpieza = Pipeline(pasos_limpieza)
df_visual = pipeline_limpieza.ajustar_transformar(datos)

print(f"df_visual: {df_visual.shape[0]} filas x {df_visual.shape[1]} columnas")
print(f"  Edad en anios : {df_visual['Edad'].min():.0f} a {df_visual['Edad'].max():.0f}")
print(f"  IMC           : {df_visual['IMC'].min():.1f} a {df_visual['IMC'].max():.1f}")

df_visual: 5511 filas x 18 columnas
  Edad en anios : 15 a 98
  IMC           : 14.8 a 63.5


## 4.2 Etiquetas legibles según el libro de códigos ENS

Los códigos numéricos de Sexo, Zona, diabetes (`di3`) y colesterol alto (`dis2`) se
traducen a texto, según `docs/diccionario_variables.csv` y el libro de códigos F1 de la
ENS 2016-2017.

In [ ]:
import numpy as np

df_visual["Sexo_txt"] = df_visual["Sexo"].map({1: "Hombre", 2: "Mujer"})
df_visual["Zona_txt"] = df_visual["Zona"].map({1: "Urbano", 2: "Rural"})
df_visual["diabetes_txt"] = df_visual["di3"].map({1: "Si", 2: "No", 3: "No recuerda"})
df_visual["colesterol_txt"] = df_visual["dis2"].map({
    1: "Si, una vez", 2: "Si, mas de una vez", 3: "Nunca", 4: "No recuerda",
})
df_visual["actividad_txt"] = df_visual["GPAQ"].map({1: "Bajo", 2: "Moderado", 3: "Alto"})
df_visual["HTA_txt"] = np.where(df_visual["HTA"] == 1, "Con HTA", "Sin HTA")

TRAMOS = [0, 30, 45, 60, 75, 120]
ETIQUETAS_EDAD = ["Menos de 30", "30 a 44", "45 a 59", "60 a 74", "75 o mas"]
df_visual["grupo_etario"] = pd.cut(df_visual["Edad"], bins=TRAMOS, labels=ETIQUETAS_EDAD, right=False)
df_visual["grupo_etario"] = df_visual["grupo_etario"].cat.set_categories(ETIQUETAS_EDAD, ordered=True)

print("df_visual listo:", df_visual.shape[1], "columnas")
df_visual[["Edad", "grupo_etario", "Sexo_txt", "Zona_txt", "HTA_txt", "diabetes_txt", "colesterol_txt", "actividad_txt"]].head(3)

df_visual listo: 25 columnas


,Edad,grupo_etario,Sexo_txt,Zona_txt,HTA_txt,diabetes_txt,colesterol_txt,actividad_txt
0,25,Menos de 30,Mujer,Urbano,Sin HTA,No,Nunca,Alto
1,20,Menos de 30,Mujer,Urbano,Sin HTA,No,Nunca,Alto
2,85,75 o mas,Mujer,Urbano,Con HTA,No,"Si, mas de una vez",Bajo


## 4.3 Ponderación por factor de expansión muestral

La ENS 2016-2017 tiene diseño muestral complejo, así que una tasa calculada directamente
sobre los 5.511 encuestados describe a la muestra, no a la población. Para que las cifras
de este informe sean comparables con las que reporta MINSAL (por ejemplo, el 27,6% de HTA
citado en la introducción), las tasas se ponderan con `Fexp_F1F2p_Corr`.

**Lo que sí y lo que no incluye esta ponderación.** Se pondera el punto estimado (la tasa),
que es lo que exige la comparabilidad con las cifras oficiales. No se estima el error
estándar ni el intervalo de confianza considerando conglomerados y estratos, eso requiere
métodos de varianza por diseño muestral que exceden el alcance de este curso, y queda
declarado como limitación en la discusión.

In [ ]:
assert "Fexp_F1F2p_Corr" in df_visual.columns, "Falta el ponderador muestral"

def resumir_por(df, variable, objetivo, ponderador="Fexp_F1F2p_Corr"):
    filas = []
    for valor, grupo in df.groupby(variable, observed=True):
        tasa = np.average(grupo[objetivo], weights=grupo[ponderador]) * 100
        filas.append({variable: valor, "personas": len(grupo), "tasa": round(tasa, 2)})
    return pd.DataFrame(filas).set_index(variable)

# Prueba rapida con HTA
print(resumir_por(df_visual, "grupo_etario", "HTA").to_string())

              personas   tasa
grupo_etario                 
Menos de 30       1109   1.85
30 a 44           1186  12.96
45 a 59           1392  41.26
60 a 74           1251  61.29
75 o mas           573  82.63
